# Thinking Data

A comprehensive guide to the data analysis lifecycle.

## Table of Contents
1. [Data Collection](#1.-Data-Collection)
2. [Data Cleaning (Pre Processing)](#2.-Data-Cleaning)
3. [Data Analysis](#3.-Data-Analysis)
4. [Data Visualization](#4.-Data-Visualization)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

---
# 1. Data Collection

Data collection is the process of gathering information from various sources for analysis. It is the **foundation** of any data-driven decision-making process.

## Why Data Collection Matters
- Sets the ceiling for analysis quality
- Even the best model cannot compensate for poor inputs
- Must be deliberate and well-documented

## Common Data Sources

| Source Type | Examples | Use Cases |
|-------------|----------|----------|
| **Files** | CSV, Excel, JSON, Parquet | Local data storage, exports |
| **Databases** | SQL, MongoDB, PostgreSQL | Enterprise data, transactions |
| **APIs** | REST, GraphQL | Real-time data, third-party services |
| **Web Scraping** | HTML parsing, BeautifulSoup | Public data, no API available |
| **Public Datasets** | Kaggle, UCI, Seaborn | Research, learning, benchmarks |

## Key Principles
1. **Document your sources** - Know where each piece of data comes from
2. **Validate data quality** - Check for completeness and accuracy
3. **Respect rate limits** - Don't overwhelm APIs or servers
4. **Consider ethics** - Ensure legal and ethical data collection

## 1.1 Reading from Files

### CSV Files
CSV (Comma-Separated Values) is the most common data format for tabular data.

In [ ]:
# Create sample CSV data
sample_data = pd.DataFrame({
    'Name': ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve', 'Frank', 'Grace'],
    'Age': [25, 30, 35, 28, 32, 45, 29],
    'City': ['New York', 'London', 'Paris', 'Tokyo', 'Sydney', 'New York', 'London'],
    'Salary': [50000, 60000, 75000, 55000, 62000, 90000, 58000],
    'Join_Date': ['2020-01-15', '2019-03-22', '2018-07-10', '2021-05-30', '2020-11-01', '2015-09-18', '2021-02-14']
})
sample_data.to_csv('sample_employees.csv', index=False)
sample_data.head()

In [ ]:
# Basic CSV reading
df = pd.read_csv('sample_employees.csv')
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()

In [ ]:
# Advanced CSV reading with options
df_advanced = pd.read_csv(
    'sample_employees.csv',
    usecols=['Name', 'Age', 'Salary'],       # Load only specific columns
    dtype={'Age': 'int32', 'Salary': 'float64'},
    na_values=['NA', 'N/A', 'null', ''],      # Values to treat as missing
    thousands=',',                             # Thousands separator
    encoding='utf-8'
)
print(df_advanced.dtypes)
df_advanced.head()

In [ ]:
# Reading large files in chunks
chunk_iter = pd.read_csv('sample_employees.csv', chunksize=3)
for i, chunk in enumerate(chunk_iter):
    print(f"Chunk {i+1}: {chunk.shape[0]} rows")

### JSON Files
JSON (JavaScript Object Notation) is commonly used for web APIs and configuration files.

In [ ]:
import json

json_data = {
    "employees": [
        {"id": 1, "name": "Alice", "dept": "Engineering", "skills": ["Python", "SQL"]},
        {"id": 2, "name": "Bob", "dept": "Marketing", "skills": ["SEO", "Analytics"]},
        {"id": 3, "name": "Charlie", "dept": "Engineering", "skills": ["Java", "AWS"]},
        {"id": 4, "name": "Diana", "dept": "HR", "skills": ["Recruiting"]},
        {"id": 5, "name": "Eve", "dept": "Finance", "skills": ["Excel", "SQL"]}
    ]
}
with open('sample_employees.json', 'w') as f:
    json.dump(json_data, f, indent=2)
print("JSON file created!")

In [ ]:
# Reading JSON files
df_json = pd.read_json('sample_employees.json')
print("Flat read:")
display(df_json.head())

# For nested JSON, use json_normalize
df_nested = pd.json_normalize(json_data['employees'])
print("\nNormalized nested JSON:")
df_nested.head()

## 1.2 Data Collection from APIs

APIs allow programmatic access to data from web services.

### Key Concepts
- **GET Request** - Retrieve data from a server
- **POST Request** - Send data to a server
- **Status Codes** - 200 (Success), 404 (Not Found), 500 (Server Error)
- **Rate Limiting** - Restrictions on how often you can request data
- **Authentication** - API keys, tokens for access control

In [ ]:
import requests

# Using JSONPlaceholder (fake REST API for testing)
response = requests.get('https://jsonplaceholder.typicode.com/users')

if response.status_code == 200:
    data = response.json()
    df_api = pd.DataFrame(data)
    print(f"Fetched {len(df_api)} records")
    df_api[['id', 'name', 'email', 'phone']].head()

In [ ]:
# API with parameters
response = requests.get(
    'https://jsonplaceholder.typicode.com/posts',
    params={'userId': 1},
    headers={'Accept': 'application/json'},
    timeout=10
)
if response.status_code == 200:
    posts = pd.DataFrame(response.json())
    print(f"Fetched {len(posts)} posts from User 1")
    posts[['id', 'title', 'body']].head()

In [ ]:
# Handling paginated APIs
def fetch_paginated(base_url, max_pages=3):
    all_data = []
    for page in range(1, max_pages + 1):
        resp = requests.get(base_url, params={'_page': page, '_limit': 10}, timeout=10)
        if resp.status_code != 200:
            break
        page_data = resp.json()
        if not page_data:
            break
        all_data.extend(page_data)
    return pd.DataFrame(all_data)

df_paginated = fetch_paginated('https://jsonplaceholder.typicode.com/comments', max_pages=3)
print(f"Total records: {len(df_paginated)}")
df_paginated.head()

## 1.3 Web Scraping

Web scraping extracts data from websites when no API is available.

**Ethical Considerations:**
- Always check `robots.txt` before scraping
- Respect rate limits and server resources
- Don't scrape personal or copyrighted data without permission

In [ ]:
from bs4 import BeautifulSoup

sample_html = """
<table id="employees">
    <thead><tr><th>Name</th><th>Department</th><th>Salary</th></tr></thead>
    <tbody>
        <tr><td>Alice</td><td>Engineering</td><td>$75,000</td></tr>
        <tr><td>Bob</td><td>Marketing</td><td>$60,000</td></tr>
        <tr><td>Charlie</td><td>Sales</td><td>$55,000</td></tr>
    </tbody>
</table>
"""

soup = BeautifulSoup(sample_html, 'html.parser')
table = soup.find('table', {'id': 'employees'})
headers = [th.text.strip() for th in table.find_all('th')]
rows = []
for tr in table.find('tbody').find_all('tr'):
    rows.append([td.text.strip() for td in tr.find_all('td')])

df_scraped = pd.DataFrame(rows, columns=headers)
df_scraped

## 1.4 Database Connections

Databases are the primary storage for enterprise data. Python connects to various databases using SQLAlchemy and specific drivers.

In [ ]:
import sqlite3
from sqlalchemy import create_engine

# Create sample database
conn = sqlite3.connect('sample_database.db')
cursor = conn.cursor()
cursor.execute('''CREATE TABLE IF NOT EXISTS products (
    id INTEGER PRIMARY KEY, name TEXT, category TEXT, price REAL, quantity INTEGER
)''')
products = [
    (1, 'Laptop', 'Electronics', 999.99, 50),
    (2, 'Mouse', 'Electronics', 29.99, 200),
    (3, 'Desk', 'Furniture', 199.99, 30),
    (4, 'Chair', 'Furniture', 149.99, 45),
    (5, 'Monitor', 'Electronics', 399.99, 75)
]
cursor.executemany('INSERT OR IGNORE INTO products VALUES (?, ?, ?, ?, ?)', products)
conn.commit()

# Read from database
df_db = pd.read_sql('SELECT * FROM products', conn)
print("Products from database:")
df_db

In [ ]:
# Using SQLAlchemy engine
engine = create_engine('sqlite:///sample_database.db')
df_filtered = pd.read_sql('SELECT * FROM products WHERE price > 100', engine)
df_filtered

## 1.5 Public Datasets

### Popular Sources
- **Kaggle** - Largest data science community
- **UCI ML Repository** - Classic ML datasets
- **Google Dataset Search** - Search engine for datasets
- **Government Data** - Data.gov, data.gov.in
- **Dataset Libraries** - Seaborn, sklearn, TensorFlow Datasets

In [ ]:
# Built-in datasets from Seaborn
print("Available datasets:", sns.get_dataset_names()[:5], "...")

df_titanic = sns.load_dataset('titanic')
print(f"Titanic: {df_titanic.shape}")
df_titanic.head()

In [ ]:
# Datasets from scikit-learn
from sklearn.datasets import load_iris

iris = load_iris()
df_iris = pd.DataFrame(iris.data, columns=iris.feature_names)
df_iris['species'] = [iris.target_names[t] for t in iris.target]
print(f"Iris: {df_iris.shape}")
df_iris.head()

## 1.6 Data Collection Summary

### Checklist
1. Identify all data sources needed
2. Document data dictionary (field names, types, meanings)
3. Check data permissions and licensing
4. Validate data completeness
5. Store raw data separately from processed data
6. Create collection scripts for reproducibility

In [ ]:
# Summary of all collected data
print("=" * 60)
print("DATA COLLECTION SUMMARY")
print("=" * 60)
print(f"1. CSV File:    {df.shape[0]} rows, {df.shape[1]} columns")
print(f"2. JSON File:   {df_json.shape[0]} rows, {df_json.shape[1]} columns")
print(f"3. API Data:    {df_api.shape[0]} rows, {df_api.shape[1]} columns")
print(f"4. Database:    {df_db.shape[0]} rows, {df_db.shape[1]} columns")
print(f"5. Titanic:     {df_titanic.shape[0]} rows, {df_titanic.shape[1]} columns")
print(f"6. Iris:        {df_iris.shape[0]} rows, {df_iris.shape[1]} columns")
print("=" * 60)